# Baseline Exact GPR Model for PM2.5 Prediction

This notebook implements a baseline Gaussian Process Regression model for predicting PM2.5 concentrations in Montana.

## Scope
- **Region**: Montana (56 monitoring sites)
- **Time period**: Single year (2019)
- **Model**: Exact GP with mini-batch training
- **Target**: PM2.5 concentration

## Covariates (based on Swanson et al.)

### Time-varying (from pm_all)
- `aot` - Aerosol Optical Depth
- `wind` - 10m wind speed (r=-0.217 with winter PM2.5)
- `hgt` - Geopotential height (r=0.211)
- `cld` - Boundary layer cloud cover (r=-0.179)
- `longwave` - Upward longwave radiation (r=0.182)
- `rh` - Relative humidity (r=-0.185)
- `tmax` - Maximum temperature
- `smogI` - Smog intensity index

### Static (from pm_fixed)
- `lat`, `lon` - Coordinates for spatial GP kernel
- `logpd2500g` - Log population density (r=0.722, strongest predictor)
- `minf_5000` - Local minima function, 5km radius (r=-0.499 to -0.605)
- `sd50k` - Terrain roughness (std elevation within 50km)
- `cadP_p12_gbm_fit_040620` - Smog potential
- `heavy_industrial_ind1` - Heavy industrial area (r=0.345)
- `housing` - Housing units

In [ ]:
import pandas as pd
import numpy as np
import torch
import gpytorch
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Import timing utilities
from timing_utils import TimingLogger, Timer

# Set random seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Initialize timing logger
timing_log = TimingLogger("gpr_timings.csv", experiment_name="baseline_random_split")
print("Timing will be logged to gpr_timings.csv")

## 1. Load and Filter Data

In [ ]:
# Load datasets
pm_all = pd.read_csv("../data/pm25_data_complete_2003-2021_nodups_051922.csv", low_memory=False)
pm_fixed = pd.read_csv('../eda/pm25_locs_with_states.csv')

print(f"pm_all: {len(pm_all):,} rows")
print(f"pm_fixed: {len(pm_fixed):,} rows")

In [ ]:
# Filter pm_fixed to Montana sites
mt_sites = pm_fixed[pm_fixed['state'] == 'MT'].copy()
print(f"Montana monitoring sites: {len(mt_sites)}")

# Get the ll_id values for Montana sites
mt_ll_ids = set(mt_sites['ll_id'].values)
print(f"Montana ll_ids: {len(mt_ll_ids)}")

In [ ]:
# Parse date and filter to 2019
pm_all['date'] = pd.to_datetime(pm_all['date'], format='%Y%m%d')
pm_all['year'] = pm_all['date'].dt.year

# Filter to Montana sites and year 2019
pm_mt = pm_all[(pm_all['ll_id'].isin(mt_ll_ids)) & (pm_all['year'] == 2019)].copy()
print(f"Montana 2019 observations: {len(pm_mt):,}")
print(f"Unique sites in 2019: {pm_mt['ll_id'].nunique()}")
print(f"Date range: {pm_mt['date'].min()} to {pm_mt['date'].max()}")

In [ ]:
# Check PM2.5 distribution
print(f"PM2.5 summary statistics:")
print(pm_mt['pm25'].describe())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(pm_mt['pm25'], bins=50, edgecolor='black')
axes[0].set_xlabel('PM2.5 (μg/m³)')
axes[0].set_ylabel('Count')
axes[0].set_title('PM2.5 Distribution')

axes[1].hist(np.log(pm_mt['pm25'] + 1), bins=50, edgecolor='black')
axes[1].set_xlabel('log(PM2.5 + 1)')
axes[1].set_ylabel('Count')
axes[1].set_title('Log-transformed PM2.5')
plt.tight_layout()
plt.show()

## 2. Select and Merge Features

In [ ]:
# Define feature sets
time_varying_features = ['aot', 'wind', 'hgt', 'cld', 'longwave', 'rh', 'tmax', 'smogI']
static_features = ['lat', 'lon', 'logpd2500g', 'minf_5000', 'sd50k', 
                   'cadP_p12_gbm_fit_040620', 'heavy_industrial_ind1', 'housing']

# Check which time-varying features are available
available_tv = [f for f in time_varying_features if f in pm_mt.columns]
missing_tv = [f for f in time_varying_features if f not in pm_mt.columns]
print(f"Available time-varying features: {available_tv}")
print(f"Missing time-varying features: {missing_tv}")

In [ ]:
# Select columns from pm_all
pm_mt_subset = pm_mt[['ll_id', 'date', 'pm25'] + available_tv].copy()

# Select static features from pm_fixed for Montana
available_static = [f for f in static_features if f in mt_sites.columns]
missing_static = [f for f in static_features if f not in mt_sites.columns]
print(f"Available static features: {available_static}")
print(f"Missing static features: {missing_static}")

mt_static = mt_sites[['ll_id'] + available_static].copy()

In [ ]:
# Merge time-varying and static data
df = pm_mt_subset.merge(mt_static, on='ll_id', how='left')
print(f"Merged dataset: {len(df):,} rows, {len(df.columns)} columns")
print(f"\nColumns: {list(df.columns)}")

In [ ]:
# Check for missing values
print("Missing values per column:")
missing = df.isnull().sum()
print(missing[missing > 0])
print(f"\nTotal rows with any missing: {df.isnull().any(axis=1).sum()}")

In [ ]:
# Define final feature list (all covariates)
feature_cols = available_tv + available_static
print(f"Feature columns ({len(feature_cols)}): {feature_cols}")

# Drop rows with missing values in features or target
df_clean = df.dropna(subset=feature_cols + ['pm25']).copy()
print(f"\nClean dataset: {len(df_clean):,} rows ({100*len(df_clean)/len(df):.1f}% retained)")

## 3. Prepare Data for GPR

In [ ]:
# Extract features and target
X = df_clean[feature_cols].values
y = df_clean['pm25'].values

# Log-transform target (PM2.5 is right-skewed)
y_log = np.log(y + 1)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"y_log range: [{y_log.min():.3f}, {y_log.max():.3f}]")

In [ ]:
# Train/test split (80/20, stratified by site to avoid data leakage)
# Using random split for now - can improve with spatial/temporal CV later
X_train, X_test, y_train, y_test = train_test_split(
    X, y_log, test_size=0.2, random_state=42
)

print(f"Training set: {len(X_train):,} samples")
print(f"Test set: {len(X_test):,} samples")

In [ ]:
# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert to PyTorch tensors
train_x = torch.tensor(X_train_scaled, dtype=torch.float32).to(device)
train_y = torch.tensor(y_train, dtype=torch.float32).to(device)
test_x = torch.tensor(X_test_scaled, dtype=torch.float32).to(device)
test_y = torch.tensor(y_test, dtype=torch.float32).to(device)

print(f"train_x: {train_x.shape}, train_y: {train_y.shape}")
print(f"test_x: {test_x.shape}, test_y: {test_y.shape}")

## 4. Define Exact GP Model

In [ ]:
class ExactGPModel(gpytorch.models.ExactGP):
    """Exact GP model with RBF kernel + constant mean."""
    
    def __init__(self, train_x, train_y, likelihood):
        super().__init__(train_x, train_y, likelihood)
        self.mean_module = gpytorch.means.ConstantMean()
        
        # RBF kernel with ARD (separate lengthscale per feature)
        self.covar_module = gpytorch.kernels.ScaleKernel(
            gpytorch.kernels.RBFKernel(ard_num_dims=train_x.shape[1])
        )
    
    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)

In [ ]:
# For exact GP with large datasets, we need to subsample
# Start with a random subset for initial training
MAX_TRAIN_SIZE = 2000  # Exact GP scales O(n³), so keep this manageable

if len(train_x) > MAX_TRAIN_SIZE:
    # Random subsample for training
    indices = torch.randperm(len(train_x))[:MAX_TRAIN_SIZE]
    train_x_subset = train_x[indices]
    train_y_subset = train_y[indices]
    print(f"Subsampled training data: {len(train_x_subset)} samples")
else:
    train_x_subset = train_x
    train_y_subset = train_y
    print(f"Using full training data: {len(train_x_subset)} samples")

In [ ]:
# Initialize likelihood and model
likelihood = gpytorch.likelihoods.GaussianLikelihood().to(device)
model = ExactGPModel(train_x_subset, train_y_subset, likelihood).to(device)

print("Model structure:")
print(model)

## 5. Train the Model

In [ ]:
# Training settings
model.train()
likelihood.train()

# Use Adam optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.1)

# Loss function: negative log marginal likelihood
mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)

n_epochs = 100
losses = []

# Time the training (hyperparameter learning)
with timing_log.time("training", 
                     n_train=len(train_x_subset), 
                     n_features=train_x_subset.shape[1],
                     n_epochs=n_epochs,
                     kernel="RBF_ARD"):
    for i in range(n_epochs):
        optimizer.zero_grad()
        output = model(train_x_subset)
        loss = -mll(output, train_y_subset)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        
        if (i + 1) % 20 == 0:
            print(f"Epoch {i+1}/{n_epochs}, Loss: {loss.item():.4f}")

print(f"Final loss: {losses[-1]:.4f}")

In [ ]:
# Plot training loss
plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Negative Log Marginal Likelihood')
plt.title('Training Loss')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Inspect learned hyperparameters
print("Learned hyperparameters:")
print(f"  Noise variance: {likelihood.noise.item():.4f}")
print(f"  Output scale: {model.covar_module.outputscale.item():.4f}")
print(f"  Lengthscales (ARD):")
lengthscales = model.covar_module.base_kernel.lengthscale.detach().cpu().numpy().flatten()
for feat, ls in zip(feature_cols, lengthscales):
    print(f"    {feat}: {ls:.4f}")

## 6. Evaluate on Test Set

In [ ]:
# Switch to evaluation mode
model.eval()
likelihood.eval()

# Make predictions with timing
batch_size = 500
means = []
variances = []

with timing_log.time("inference", 
                     n_train=len(train_x_subset),
                     n_test=len(test_x), 
                     n_features=test_x.shape[1],
                     batch_size=batch_size):
    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        for i in range(0, len(test_x), batch_size):
            batch_x = test_x[i:i+batch_size]
            pred = likelihood(model(batch_x))
            means.append(pred.mean.cpu())
            variances.append(pred.variance.cpu())
        
        pred_mean = torch.cat(means).numpy()
        pred_var = torch.cat(variances).numpy()

print(f"Predictions shape: {pred_mean.shape}")

In [ ]:
# Calculate metrics on log scale
y_test_np = test_y.cpu().numpy()

# RMSE on log scale
rmse_log = np.sqrt(np.mean((pred_mean - y_test_np)**2))

# R² on log scale
ss_res = np.sum((y_test_np - pred_mean)**2)
ss_tot = np.sum((y_test_np - np.mean(y_test_np))**2)
r2_log = 1 - (ss_res / ss_tot)

# MAE on log scale
mae_log = np.mean(np.abs(pred_mean - y_test_np))

print("Metrics on log(PM2.5 + 1) scale:")
print(f"  RMSE: {rmse_log:.4f}")
print(f"  MAE:  {mae_log:.4f}")
print(f"  R²:   {r2_log:.4f}")

In [ ]:
# Transform back to original scale
pred_pm25 = np.exp(pred_mean) - 1
actual_pm25 = np.exp(y_test_np) - 1

# Metrics on original scale
rmse_orig = np.sqrt(np.mean((pred_pm25 - actual_pm25)**2))
mae_orig = np.mean(np.abs(pred_pm25 - actual_pm25))
ss_res_orig = np.sum((actual_pm25 - pred_pm25)**2)
ss_tot_orig = np.sum((actual_pm25 - np.mean(actual_pm25))**2)
r2_orig = 1 - (ss_res_orig / ss_tot_orig)

print("\nMetrics on original PM2.5 scale (μg/m³):")
print(f"  RMSE: {rmse_orig:.2f}")
print(f"  MAE:  {mae_orig:.2f}")
print(f"  R²:   {r2_orig:.4f}")

In [ ]:
# Plot predictions vs actual
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Log scale
axes[0].scatter(y_test_np, pred_mean, alpha=0.3, s=10)
axes[0].plot([y_test_np.min(), y_test_np.max()], 
             [y_test_np.min(), y_test_np.max()], 'r--', lw=2)
axes[0].set_xlabel('Actual log(PM2.5 + 1)')
axes[0].set_ylabel('Predicted log(PM2.5 + 1)')
axes[0].set_title(f'Log Scale (R² = {r2_log:.3f})')
axes[0].grid(True, alpha=0.3)

# Original scale
axes[1].scatter(actual_pm25, pred_pm25, alpha=0.3, s=10)
max_val = max(actual_pm25.max(), pred_pm25.max())
axes[1].plot([0, max_val], [0, max_val], 'r--', lw=2)
axes[1].set_xlabel('Actual PM2.5 (μg/m³)')
axes[1].set_ylabel('Predicted PM2.5 (μg/m³)')
axes[1].set_title(f'Original Scale (R² = {r2_orig:.3f})')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Residual analysis
residuals = pred_pm25 - actual_pm25

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Residual distribution
axes[0].hist(residuals, bins=50, edgecolor='black')
axes[0].axvline(x=0, color='r', linestyle='--')
axes[0].set_xlabel('Residual (μg/m³)')
axes[0].set_ylabel('Count')
axes[0].set_title('Residual Distribution')

# Residuals vs predicted
axes[1].scatter(pred_pm25, residuals, alpha=0.3, s=10)
axes[1].axhline(y=0, color='r', linestyle='--')
axes[1].set_xlabel('Predicted PM2.5 (μg/m³)')
axes[1].set_ylabel('Residual (μg/m³)')
axes[1].set_title('Residuals vs Predicted')

plt.tight_layout()
plt.show()

print(f"Residual statistics:")
print(f"  Mean: {residuals.mean():.3f}")
print(f"  Std:  {residuals.std():.3f}")
print(f"  Min:  {residuals.min():.3f}")
print(f"  Max:  {residuals.max():.3f}")

## 7. Feature Importance (via ARD Lengthscales)

In ARD (Automatic Relevance Determination), shorter lengthscales indicate more important features.

In [ ]:
# Feature importance from ARD lengthscales
# Smaller lengthscale = more important (function varies more rapidly with that feature)
importance = 1.0 / lengthscales
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'lengthscale': lengthscales,
    'importance': importance
}).sort_values('importance', ascending=False)

print("Feature importance (1/lengthscale):")
print(importance_df.to_string(index=False))

In [ ]:
# Plot feature importance
plt.figure(figsize=(10, 6))
plt.barh(importance_df['feature'], importance_df['importance'])
plt.xlabel('Importance (1/lengthscale)')
plt.ylabel('Feature')
plt.title('Feature Importance from ARD Lengthscales')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 8. Summary and Next Steps

### Model Summary
- Exact GP with RBF kernel and ARD
- Trained on subset of Montana 2019 data
- Log-transformed PM2.5 target

### Potential Improvements
1. **Spatial cross-validation**: Use leave-one-site-out or spatial blocking to avoid data leakage
2. **Temporal features**: Add day-of-year, month, or seasonal indicators
3. **More covariates**: LST climatology, additional terrain features
4. **Kernel exploration**: Try Matérn kernel, periodic kernels for seasonality
5. **Sparse GP**: Use inducing points to scale to full dataset
6. **Ensemble**: Combine multiple GPs trained on different subsets

In [ ]:
# Save model state for later use
torch.save({
    'model_state_dict': model.state_dict(),
    'likelihood_state_dict': likelihood.state_dict(),
    'scaler_mean': scaler.mean_,
    'scaler_scale': scaler.scale_,
    'feature_cols': feature_cols,
}, 'baseline_gpr_model.pt')

print("Model saved to baseline_gpr_model.pt")

# Display timing summary
timing_log.summary()